# AQI Prediction Model — Air Quality Monitoring Project
**Role: Machine Learning Lead (Phase 5)**

This notebook downloads **historical PM2.5 data for Dhaka from the OpenAQ v3 API**, trains a model, then compares its **predicted** PM2.5 / AQI against the **actual** recorded values held out in a test set.

**Pipeline:** Download (OpenAQ API) → Load → Clean → Compute AQI → Feature engineering → Time-based split → Train (Linear Regression + Random Forest) → Compare predicted vs actual → Plots & metrics.

## 0. Imports

In [1]:
import os, time, json, datetime as dt
import numpy as np
import pandas as pd
import requests
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True

## 1. Download data from the OpenAQ v3 API

OpenAQ retired the v1 and v2 APIs on **31 January 2025**, so we use **v3**, which needs a free API key.

**One-time setup**
1. Register at [explore.openaq.org/register](https://explore.openaq.org/register) and copy your key from [explore.openaq.org/account](https://explore.openaq.org/account).
2. Paste it into `API_KEY` below (or set an `OPENAQ_API_KEY` environment variable).
3. **Finding your station is automatic:** leave `LOCATION_ID = 0` and the notebook picks the nearest Dhaka PM2.5 station for you (and prints the others so you can switch). To choose manually instead, open [explore.openaq.org](https://explore.openaq.org), click a station, and read the id from the URL.

OpenAQ's data model is **location → sensors → measurements** (one sensor per pollutant), so we find the location's PM2.5 sensor, then pull its hourly history.

> Set `DOWNLOAD_FROM_API = False` to skip the API and use a local CSV you already have.

### 1a. Config

In [2]:
DOWNLOAD_FROM_API = False            # False = use an existing local CSV instead

API_KEY  = os.environ.get('OPENAQ_API_KEY', '17803b50a1dac87cafdb7056be122a9e9ea8d1e8abef8df48f3183c8fb2a8d1c')
BASE     = 'https://api.openaq.org/v3'

DHAKA_LAT, DHAKA_LON = 23.8103, 90.4125
SEARCH_RADIUS_M = 25000             # max 25 km

LOCATION_ID = 0                     # 0 = auto-pick nearest Dhaka PM2.5 station; or set your own id
START   = '2024-01-01'              # inclusive (UTC date)
END     = '2025-01-01'              # upper bound
OUT_CSV = 'openaq_dhaka_pm25.csv'

PM25_PARAMETER_ID = 2               # OpenAQ parameter id for pm25
HEADERS = {'X-API-Key': API_KEY}

# Fallback file used when DOWNLOAD_FROM_API = False:
DATA_PATH = '../dataset/training_data.csv'

### 1b. Helper functions

In [3]:
def get(path, params=None):
    """GET a v3 endpoint with simple 429 (rate-limit) back-off."""
    for attempt in range(6):
        r = requests.get(f'{BASE}{path}', headers=HEADERS, params=params, timeout=60)
        if r.status_code == 429:
            wait = int(r.headers.get('Retry-After', 2 ** attempt))
            print(f'  rate limited, waiting {wait}s...'); time.sleep(wait); continue
        if r.status_code == 401:
            raise SystemExit('401 Unauthorized - check your API_KEY.')
        r.raise_for_status()
        return r.json()
    raise SystemExit('Gave up after repeated rate-limit responses.')

def list_pm25_locations():
    """Return PM2.5 stations near Dhaka, nearest first, with last-reporting date."""
    data = get('/locations', {'coordinates': f'{DHAKA_LAT},{DHAKA_LON}',
                              'radius': SEARCH_RADIUS_M,
                              'parameters_id': PM25_PARAMETER_ID, 'limit': 100})
    locs = data.get('results', [])
    locs.sort(key=lambda l: l.get('distance', 1e12))   # nearest first when available
    print(f"{'id':>9}  {'dist(km)':>8}  {'last report':>11}  name"); print('-' * 70)
    for l in locs:
        d = l.get('distance'); d = f'{d/1000:8.1f}' if isinstance(d,(int,float)) else '   n/a  '
        last = ((l.get('datetimeLast') or {}).get('utc') or 'unknown')[:10]
        print(f"{l['id']:>9}  {d}  {last:>11}  {l.get('name')}")
    return locs

def sensor_coverage(sensor_id):
    """Return (first_utc, last_utc) ISO strings for what a sensor actually has."""
    s = get(f'/sensors/{sensor_id}')['results'][0]
    return ((s.get('datetimeFirst') or {}).get('utc'),
            (s.get('datetimeLast')  or {}).get('utc'))

def find_pm25_sensor(location_id):
    """Return the sensor id that measures pm25 at this location."""
    data = get(f'/locations/{location_id}/sensors', {'limit': 100})
    for s in data.get('results', []):
        if (s.get('parameter') or {}).get('name') == 'pm25':
            print(f"Found PM2.5 sensor id={s['id']} at location {location_id}"); return s['id']
    raise SystemExit('No pm25 sensor here. Parameters present: ' +
                     str([(s.get('parameter') or {}).get('name') for s in data.get('results', [])]))

def month_chunks(start, end):
    """Yield (from, to) ISO date strings one month at a time."""
    cur, end_d = dt.date.fromisoformat(start), dt.date.fromisoformat(end)
    while cur < end_d:
        nxt = min((cur.replace(day=1) + dt.timedelta(days=32)).replace(day=1), end_d)
        yield cur.isoformat(), nxt.isoformat(); cur = nxt

def download_hourly(sensor_id, start, end):
    """Pull precomputed hourly values for a sensor across a date range."""
    rows = []
    for dfrom, dto in month_chunks(start, end):
        page = 1
        while True:
            data = get(f'/sensors/{sensor_id}/hours',
                       {'datetime_from': dfrom, 'datetime_to': dto, 'limit': 1000, 'page': page})
            results = data.get('results', [])
            for item in results:
                ts = ((item.get('period') or {}).get('datetimeFrom') or {}).get('utc')
                rows.append({'datetime': ts, 'pm25': item.get('value')})
            print(f'  {dfrom} -> {dto}  page {page}: {len(results)} rows')
            if len(results) < 1000: break
            page += 1; time.sleep(0.2)
    return pd.DataFrame(rows)

### 1c. Run the download
With `LOCATION_ID = 0`, this lists the Dhaka PM2.5 stations (with each one's **last-reporting date**, so you can avoid inactive stations), auto-selects the nearest, and downloads its hourly history. It automatically **clamps the date range to what the chosen sensor actually has**, so an old station won't silently return zero rows. To pick a different station, copy an id from the printed list into `LOCATION_ID` (Section 1a) and rerun.

In [4]:
if DOWNLOAD_FROM_API:
    assert API_KEY != 'PASTE-YOUR-KEY-HERE', 'Set API_KEY (or OPENAQ_API_KEY) first.'

    if not LOCATION_ID:                       # auto-pick nearest Dhaka PM2.5 station
        candidates = list_pm25_locations()
        assert candidates, 'No PM2.5 stations found near Dhaka - widen SEARCH_RADIUS_M.'
        LOCATION_ID = candidates[0]['id']
        print(f"\nAuto-selected LOCATION_ID = {LOCATION_ID} ({candidates[0].get('name')})")
        print('(Set LOCATION_ID in Section 1a to a different id above if you prefer.)\n')

    sensor_id = find_pm25_sensor(LOCATION_ID)

    # Clamp the requested dates to what this sensor actually has, so an
    # inactive/old station doesn't silently return zero rows.
    first_utc, last_utc = sensor_coverage(sensor_id)
    print(f'Sensor {sensor_id} reports {first_utc} -> {last_utc}')
    eff_start, eff_end = START, END
    if first_utc and last_utc:
        fdate = first_utc[:10]
        ldate = (pd.to_datetime(last_utc) + pd.Timedelta(days=1)).date().isoformat()
        eff_start, eff_end = max(START, fdate), min(END, ldate)
        if eff_start >= eff_end:        # requested window misses coverage entirely
            eff_start, eff_end = fdate, ldate
            print('Requested dates are outside coverage - using full available range.')
    print(f'Downloading {eff_start} -> {eff_end}')
    raw = download_hourly(sensor_id, eff_start, eff_end)
    assert not raw.empty, 'Still no data - try a different station (set LOCATION_ID=0 to auto-pick an active one).'
    raw['datetime'] = pd.to_datetime(raw['datetime'], utc=True, errors='coerce')
    raw = raw.dropna(subset=['datetime']).sort_values('datetime').drop_duplicates('datetime')
    raw.to_csv(OUT_CSV, index=False)
    DATA_PATH = OUT_CSV
    print(f'\nSaved {len(raw)} hourly rows to {OUT_CSV}')
    print('Range:', raw['datetime'].min(), '->', raw['datetime'].max())
else:
    print('Skipping API download. Using local file:', DATA_PATH)

Skipping API download. Using local file: /root/files/dataset/training_data.csv


## 2. Load the data
Reads `DATA_PATH` (the file just downloaded, or your own). The loader auto-detects the timestamp column and, if the file is in OpenAQ's *long* format (a `parameter`/`value` layout), pivots it to columns.

In [5]:
df = pd.read_csv(DATA_PATH)
print('Raw shape:', df.shape); print('Columns:', list(df.columns))

DT_CANDIDATES = {'datetimeutc','datetime','date_utc','utc','date','local','datetimelocal','datetimelocaltime'}
dt_col = next((c for c in df.columns if c.lower() in DT_CANDIDATES), None)
assert dt_col is not None, f'No datetime column found in {list(df.columns)}'
df[dt_col] = pd.to_datetime(df[dt_col], errors='coerce', utc=True)

if {'parameter','value'}.issubset(df.columns):
    df = (df.pivot_table(index=dt_col, columns='parameter', values='value', aggfunc='mean')
            .reset_index().rename_axis(None, axis=1))
    print('Pivoted long -> wide. Pollutant columns:', [c for c in df.columns if c != dt_col])

df = df.rename(columns={dt_col: 'datetime'}).sort_values('datetime')
assert 'pm25' in df.columns, f"Expected a 'pm25' column. Found: {list(df.columns)}"
df[['datetime','pm25']].head()

Raw shape: (4280, 3)
Columns: ['datetime', 'pm25', 'aqi']


,datetime,pm25
0,2025-12-09 10:00:00+00:00,67.7
1,2025-12-09 11:00:00+00:00,65.2
2,2025-12-09 12:00:00+00:00,77.0
3,2025-12-09 13:00:00+00:00,102.0
4,2025-12-09 14:00:00+00:00,144.0


## 3. Clean the data
Remove sensor error codes, put the data on a regular hourly grid, clip extreme outliers (IQR rule), and fill short gaps.

In [6]:
df['pm25'] = pd.to_numeric(df['pm25'], errors='coerce')
df.loc[(df['pm25'] < 0) | (df['pm25'] > 1000), 'pm25'] = np.nan   # -999 etc. are error codes

df = df.dropna(subset=['datetime']).set_index('datetime')
df = df[~df.index.duplicated(keep='first')]
df = df.resample('h').mean(numeric_only=True)

q1, q3 = df['pm25'].quantile([0.25, 0.75]); iqr = q3 - q1
df['pm25'] = df['pm25'].clip(q1 - 3*iqr, q3 + 3*iqr)
df['pm25'] = df['pm25'].interpolate(limit=6).ffill().bfill()

print('Clean shape:', df.shape, '| remaining nulls in pm25:', int(df['pm25'].isna().sum()))
df['pm25'].describe()

Clean shape: (4280, 2) | remaining nulls in pm25: 0


count    4280.000000
mean      127.604451
std        88.689559
min         3.600000
25%        52.300000
50%        95.700000
75%       194.000000
max       601.000000
Name: pm25, dtype: float64

## 4. Quick EDA
Sanity-check the series and the daily pollution cycle (useful figures for the report).

In [7]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df['pm25'].plot(ax=ax[0], lw=0.6); ax[0].set_title('PM2.5 over time'); ax[0].set_ylabel('µg/m³')
df.groupby(df.index.hour)['pm25'].mean().plot(ax=ax[1], marker='o')
ax[1].set_title('Average PM2.5 by hour of day'); ax[1].set_xlabel('Hour'); ax[1].set_ylabel('µg/m³')
plt.tight_layout(); plt.savefig('eda_overview.png', dpi=120); plt.show()

/tmp/ipykernel_39883/3212287079.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig('eda_overview.png', dpi=120); plt.show()


## 5. Compute AQI from PM2.5
AQI is derived from PM2.5 using the **US EPA piecewise-linear formula** with the **2024 revised breakpoints** (effective 6 May 2024):

$$AQI = \frac{I_{hi}-I_{lo}}{C_{hi}-C_{lo}}\,(C - C_{lo}) + I_{lo}$$

| PM2.5 (µg/m³) | AQI | Category |
|---|---|---|
| 0.0 – 9.0 | 0 – 50 | Good |
| 9.1 – 35.4 | 51 – 100 | Moderate |
| 35.5 – 55.4 | 101 – 150 | Unhealthy for Sensitive Groups |
| 55.5 – 125.4 | 151 – 200 | Unhealthy |
| 125.5 – 225.4 | 201 – 300 | Very Unhealthy |
| 225.5 – 325.4 | 301 – 500 | Hazardous |

> Official AQI uses a 24-hour average PM2.5. Set `USE_24H = True` to use the 24-hour rolling mean (more standards-correct).

In [8]:
BREAKPOINTS = [(0.0, 9.0, 0, 50), (9.1, 35.4, 51, 100), (35.5, 55.4, 101, 150),
               (55.5, 125.4, 151, 200), (125.5, 225.4, 201, 300), (225.5, 325.4, 301, 500)]

def pm_to_aqi(c):
    if pd.isna(c): return np.nan
    c = np.floor(c * 10) / 10            # EPA truncates PM2.5 to 0.1 µg/m³
    for clo, chi, ilo, ihi in BREAKPOINTS:
        if clo <= c <= chi:
            return round((ihi - ilo) / (chi - clo) * (c - clo) + ilo)
    return 500

USE_24H = False
pm_for_aqi = df['pm25'].rolling(24, min_periods=1).mean() if USE_24H else df['pm25']
df['aqi'] = pm_for_aqi.apply(pm_to_aqi)
df[['pm25','aqi']].head()

,pm25,aqi
datetime,,
2025-12-09 10:00:00+00:00,67.7,160
2025-12-09 11:00:00+00:00,65.2,158
2025-12-09 12:00:00+00:00,77.0,166
2025-12-09 13:00:00+00:00,102.0,184
2025-12-09 14:00:00+00:00,144.0,219


## 6. Feature engineering
Predict the **next** PM2.5 from time-of-day signals and recent history. Temperature / humidity columns are used automatically if present in your file.

In [9]:
df['hour']  = df.index.hour
df['dow']   = df.index.dayofweek
df['month'] = df.index.month
for lag in [1, 2, 3, 24]:
    df[f'pm25_lag{lag}'] = df['pm25'].shift(lag)
df['pm25_roll6']  = df['pm25'].shift(1).rolling(6).mean()
df['pm25_roll24'] = df['pm25'].shift(1).rolling(24).mean()

optional = [c for c in ['temperature','humidity','temp','rh'] if c in df.columns]
FEATURES = ['hour','dow','month','pm25_lag1','pm25_lag2','pm25_lag3','pm25_lag24',
            'pm25_roll6','pm25_roll24'] + optional

model_df = df.dropna(subset=FEATURES + ['pm25']).copy()
X = model_df[FEATURES]; y = model_df['pm25']
print('Features:', FEATURES); print('Samples for modelling:', len(model_df))

Features: ['hour', 'dow', 'month', 'pm25_lag1', 'pm25_lag2', 'pm25_lag3', 'pm25_lag24', 'pm25_roll6', 'pm25_roll24']
Samples for modelling: 4256


## 7. Train / test split (time-based)
For a time series we **must not shuffle** — train on the earlier 80%, test on the most recent 20% the model has never seen. Those test rows are our **actual** values.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
test_index = model_df.index[-len(X_test):]
print(f'Train: {len(X_train)} rows  ({model_df.index[0].date()} -> {model_df.index[len(X_train)-1].date()})')
print(f'Test : {len(X_test)} rows  ({test_index[0].date()} -> {test_index[-1].date()})')

Train: 3404 rows  (2025-12-10 -> 2026-05-01)
Test : 852 rows  (2026-05-01 -> 2026-06-05)


## 8. Train models and compare predicted vs actual
Two models, matching the project's *Easy* and *Medium* suggestions. Metrics are reported for **both PM2.5 and the derived AQI**.

In [11]:
def evaluate(model, name):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2   = r2_score(y_test, pred)
    aqi_actual = y_test.apply(pm_to_aqi).values
    aqi_pred   = pd.Series(pred, index=y_test.index).apply(pm_to_aqi).values
    amae  = mean_absolute_error(aqi_actual, aqi_pred)
    armse = np.sqrt(mean_squared_error(aqi_actual, aqi_pred))
    return {'model': name, 'pred': pred, 'aqi_pred': aqi_pred, 'aqi_actual': aqi_actual,
            'PM2.5 MAE': mae, 'PM2.5 RMSE': rmse, 'PM2.5 R2': r2,
            'AQI MAE': amae, 'AQI RMSE': armse, 'fitted': model}

results = [evaluate(LinearRegression(), 'Linear Regression'),
           evaluate(RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), 'Random Forest')]

metrics = pd.DataFrame(results)[['model','PM2.5 MAE','PM2.5 RMSE','PM2.5 R2','AQI MAE','AQI RMSE']]
metrics = metrics.round(3).set_index('model')
metrics

,PM2.5 MAE,PM2.5 RMSE,PM2.5 R2,AQI MAE,AQI RMSE
model,,,,,
Linear Regression,7.454,11.883,0.738,11.729,17.029
Random Forest,8.050,12.530,0.708,12.474,17.971


## 9. Actual vs Predicted plots
The core deliverable. We plot the better-performing model (lower PM2.5 RMSE).

In [12]:
best = min(results, key=lambda r: r['PM2.5 RMSE'])
print('Best model:', best['model'])

plt.figure(figsize=(13, 4))
plt.plot(test_index, y_test.values, label='Actual PM2.5', lw=1)
plt.plot(test_index, best['pred'], label='Predicted PM2.5', lw=1, alpha=0.8)
plt.title(f"Actual vs Predicted PM2.5 - {best['model']}"); plt.ylabel('µg/m³'); plt.legend()
plt.tight_layout(); plt.savefig('actual_vs_pred_pm25.png', dpi=120); plt.show()

plt.figure(figsize=(13, 4))
plt.plot(test_index, best['aqi_actual'], label='Actual AQI', lw=1)
plt.plot(test_index, best['aqi_pred'], label='Predicted AQI', lw=1, alpha=0.8)
plt.title(f"Actual vs Predicted AQI - {best['model']}"); plt.ylabel('AQI'); plt.legend()
plt.tight_layout(); plt.savefig('actual_vs_pred_aqi.png', dpi=120); plt.show()

Best model: Linear Regression


/tmp/ipykernel_39883/4207451901.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig('actual_vs_pred_pm25.png', dpi=120); plt.show()


/tmp/ipykernel_39883/4207451901.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig('actual_vs_pred_aqi.png', dpi=120); plt.show()


In [13]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for a, lab in [(ax[0], 'PM2.5'), (ax[1], 'AQI')]:
    actual = y_test.values if lab=='PM2.5' else best['aqi_actual']
    pred   = best['pred'] if lab=='PM2.5' else best['aqi_pred']
    a.scatter(actual, pred, s=6, alpha=0.4)
    lims = [min(actual.min(), pred.min()), max(actual.max(), pred.max())]
    a.plot(lims, lims, 'r--', lw=1); a.set_xlabel(f'Actual {lab}'); a.set_ylabel(f'Predicted {lab}')
    a.set_title(f'{lab}: predicted vs actual')
plt.tight_layout(); plt.savefig('scatter_pred_vs_actual.png', dpi=120); plt.show()

/tmp/ipykernel_39883/3960555299.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig('scatter_pred_vs_actual.png', dpi=120); plt.show()


In [14]:
rf = next((r['fitted'] for r in results if r['model']=='Random Forest'), None)
if rf is not None:
    imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
    imp.plot(kind='barh', figsize=(8,5)); plt.title('Random Forest feature importance')
    plt.tight_layout(); plt.savefig('feature_importance.png', dpi=120); plt.show()

/tmp/ipykernel_39883/178037306.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig('feature_importance.png', dpi=120); plt.show()


## 10. Save outputs
Writes the metrics table and a per-row actual-vs-predicted CSV — ready for the report and the dashboard team.

In [15]:
metrics.to_csv('model_metrics.csv')
out = pd.DataFrame({'datetime': test_index,
                    'actual_pm25': y_test.values, 'predicted_pm25': best['pred'],
                    'actual_aqi': best['aqi_actual'], 'predicted_aqi': best['aqi_pred']})
out.to_csv('predictions_actual_vs_predicted.csv', index=False)
print('Saved: model_metrics.csv, predictions_actual_vs_predicted.csv')
out.head()

Saved: model_metrics.csv, predictions_actual_vs_predicted.csv


,datetime,actual_pm25,predicted_pm25,actual_aqi,predicted_aqi
0,2026-05-01 06:00:00+00:00,28.3,41.476476,87,116
1,2026-05-01 07:00:00+00:00,37.7,30.277971,106,90
2,2026-05-01 08:00:00+00:00,38.4,41.905672,108,117
3,2026-05-01 09:00:00+00:00,45.8,39.183855,126,110
4,2026-05-01 10:00:00+00:00,45.5,47.611081,126,131


## 11. Export for the web dashboard
Saves everything the Flask app needs: the trained model, the full hourly series (with AQI), and the metrics. Drop these files next to `app.py`.

In [16]:
# 1) best model + the feature order it expects
joblib.dump({'model': best['fitted'], 'features': FEATURES}, 'aqi_model.joblib')

# 2) full cleaned hourly series with AQI (for the trend chart + forecasting)
df[['pm25','aqi']].reset_index().rename(columns={'index':'datetime'}).to_csv('dashboard_data.csv', index=False)

# 3) headline metrics for the dashboard cards
with open('metrics.json', 'w') as f:
    json.dump({'best_model': best['model'],
               'pm25_mae': round(float(best['PM2.5 MAE']), 2),
               'pm25_rmse': round(float(best['PM2.5 RMSE']), 2),
               'pm25_r2': round(float(best['PM2.5 R2']), 3),
               'aqi_mae': round(float(best['AQI MAE']), 2)}, f, indent=2)

print('Exported: aqi_model.joblib, dashboard_data.csv, metrics.json')
print('(predictions_actual_vs_predicted.csv from Section 10 is also used by the app.)')

Exported: aqi_model.joblib, dashboard_data.csv, metrics.json
(predictions_actual_vs_predicted.csv from Section 10 is also used by the app.)


---
### How to read the results
- **MAE** — average error in the same units (µg/m³ for PM2.5, points for AQI). Lower is better.
- **RMSE** — like MAE but punishes large misses harder.
- **R²** — share of variation explained; 1.0 is perfect, ~0 is no better than the mean.

### For your report (Phase 5 deliverables)
1. State that, because calibrated sensors were out of budget, the model was trained and validated on **public OpenAQ Dhaka historical data (v3 API)** — a standard, defensible approach.
2. Include the **metrics table** (Section 8) and the **actual-vs-predicted plots** (Section 9).
3. Note that **AQI is a deterministic function of PM2.5** (EPA 2024 formula), so predicting PM2.5 well gives AQI for free.
4. Possible improvements: more history, weather features (temperature/humidity), trying XGBoost or an LSTM.